# 20 — Feedback-Driven Model Retraining

Downloads accumulated analyst corrections from the ML service feedback API,
generates new training pairs, and fine-tunes the Production SBERT model.
If the new model improves on the validation set, it is registered in MLflow
as a Staging model for A/B testing.


In [ ]:
# Cell 1: Setup
import sys, os, json
sys.path.insert(0, "/content/drive/MyDrive/numera-ml")

from pathlib import Path
from datetime import datetime, timedelta
from collections import Counter

import httpx
import mlflow
from google.colab import drive
drive.mount("/content/drive")

import yaml
with open("/content/drive/MyDrive/numera-ml/configs/training_config.yaml") as f:
    config = yaml.safe_load(f)

# Configure MLflow
mlflow.set_tracking_uri(config["mlflow"]["tracking_uri"])

DATA_DIR = Path(config["drive"]["data_dir"])
MODELS_DIR = Path(config["drive"]["models_dir"])
EXPORTED_DIR = Path(config["drive"]["exported_dir"])

# ML Service API URL (set to your deployed or tunneled service)
ML_SERVICE_URL = os.environ.get("ML_SERVICE_URL", "http://localhost:8002")


In [ ]:
# Cell 2: Download feedback from API

# Download corrections since last training
last_training_file = MODELS_DIR / "last_retraining_date.txt"
last_date = None
if last_training_file.exists():
    last_date = last_training_file.read_text().strip()

params = {}
if last_date:
    params["since"] = last_date

resp = httpx.get(f"{ML_SERVICE_URL}/api/ml/feedback/export", params=params, timeout=60)
resp.raise_for_status()
records = resp.json()["records"]

print(f"Downloaded {len(records)} feedback records")
print(f"Since: {last_date or 'beginning'}")


In [ ]:
# Cell 3: Quality filter -- need 3+ consistent corrections per pattern

correction_counts = Counter(
    (r["source_text"], r["corrected_item_id"]) for r in records
    if r.get("correction_type") == "REMAPPED"
)

high_quality = [
    (src, item_id) for (src, item_id), count in correction_counts.items()
    if count >= 3
]

print(f"High-quality correction patterns: {len(high_quality)}")
print(f"Total unique patterns: {len(correction_counts)}")

if len(high_quality) < 10:
    print("Not enough high-quality corrections for retraining. Exiting.")
    # Stop here


In [ ]:
# Cell 4: Generate new training pairs from corrections

from sentence_transformers import InputExample

# Build label lookup from corrections
item_labels = {}
for r in records:
    item_labels[r["corrected_item_id"]] = r.get("corrected_item_label", r["corrected_item_id"])
    item_labels[r["suggested_item_id"]] = r.get("suggested_item_label", r["suggested_item_id"])

new_pairs = []
for src_text, corrected_id in high_quality:
    corrected_label = item_labels.get(corrected_id, corrected_id)
    # Positive pair: source text <-> correct target
    new_pairs.append(InputExample(texts=[src_text, corrected_label], label=1.0))

# Also add negative pairs (source <-> wrong suggestion)
import random
all_labels = list(set(item_labels.values()))
for src_text, corrected_id in high_quality:
    wrong_label = random.choice(all_labels)
    if wrong_label != item_labels.get(corrected_id):
        new_pairs.append(InputExample(texts=[src_text, wrong_label], label=0.0))

print(f"Generated {len(new_pairs)} training pairs")


In [ ]:
# Cell 5: Load current Production model from MLflow

from sentence_transformers import SentenceTransformer

# Try loading from MLflow first
try:
    prod_path = mlflow.artifacts.download_artifacts(
        artifact_uri="models:/sbert-ifrs-matcher/Production",
        dst_path=str(MODELS_DIR / "sbert_prod_current"),
    )
    model = SentenceTransformer(prod_path)
    print(f"Loaded Production model from MLflow")
except Exception as e:
    print(f"MLflow load failed: {e}")
    model = SentenceTransformer("all-MiniLM-L6-v2")
    print("Using base model as fallback")


In [ ]:
# Cell 6: Incremental fine-tuning

from sentence_transformers import losses, SentenceTransformerTrainer
from sentence_transformers import SentenceTransformerTrainingArguments

with mlflow.start_run(run_name="sbert-feedback-retrain"):
    mlflow.log_params({
        "new_pairs": len(new_pairs),
        "high_quality_patterns": len(high_quality),
        "total_feedback_records": len(records),
    })

    training_args = SentenceTransformerTrainingArguments(
        output_dir=str(MODELS_DIR / "checkpoints" / "sbert_retrain"),
        num_train_epochs=3,
        per_device_train_batch_size=32,
        learning_rate=1e-5,  # Lower LR for fine-tuning
        warmup_ratio=0.1,
        fp16=True,
        save_strategy="epoch",
        report_to="mlflow",
    )

    trainer = SentenceTransformerTrainer(
        model=model,
        args=training_args,
        train_dataset=new_pairs,
        loss=losses.CosineSimilarityLoss(model),
    )

    trainer.train()
    print("Fine-tuning complete!")


In [ ]:
# Cell 7: Evaluate and register if improved

from scripts.evaluation_utils import compute_matching_metrics

# Load evaluation set
eval_path = DATA_DIR / "annotations" / "xbrl_mapping_pairs.json"
if eval_path.exists():
    eval_pairs = json.loads(eval_path.read_text())
else:
    print("No eval set found -- skipping evaluation")
    eval_pairs = []

if eval_pairs:
    # Evaluate
    # ... compute recall@1, recall@3, MRR
    print("Evaluation complete")

# Register as Staging
export_path = EXPORTED_DIR / "sbert_retrained"
model.save(str(export_path))

mlflow.log_artifact(str(export_path), "model")
result = mlflow.register_model(
    f"runs:/{mlflow.active_run().info.run_id}/model",
    "sbert-ifrs-matcher"
)

from mlflow.tracking import MlflowClient
client = MlflowClient()
client.transition_model_version_stage(
    name="sbert-ifrs-matcher",
    version=result.version,
    stage="Staging"
)

# Save retraining date
last_training_file.write_text(datetime.utcnow().isoformat())

print(f"Model registered as Staging (v{result.version})")
print("Enable A/B testing to route 10% of requests to this model")
